In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
START = "2021-01-01"
END = "2025-12-31"


def download_adj_close(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    end_exclusive = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    data = yf.download(tickers, start=start, end=end_exclusive, auto_adjust=False, progress=False)
    prices = data["Adj Close"][tickers].dropna()
    if prices.empty:
        raise ValueError("No price data downloaded.")
    return prices


def compute_log_returns(prices: pd.DataFrame) -> pd.DataFrame:
    return np.log(prices / prices.shift(1)).dropna()


def equal_weights(tickers: list[str]) -> pd.Series:
    return pd.Series(1 / len(tickers), index=tickers)


def compute_portfolio_returns(log_returns: pd.DataFrame, weights: pd.Series) -> pd.Series:
    simple_returns = np.expm1(log_returns)
    return np.log1p(simple_returns @ weights).rename("Portfolio")


def prepare_portfolio_returns(tickers: list[str], start: str, end: str) -> pd.Series:
    prices = download_adj_close(tickers, start, end)
    log_returns = compute_log_returns(prices)
    weights = equal_weights(tickers)
    return compute_portfolio_returns(log_returns, weights)


portfolio_returns = prepare_portfolio_returns(TICKERS, START, END)

prices_df = download_adj_close(TICKERS, START, END)
log_returns_df = compute_log_returns(prices_df)
weights = equal_weights(TICKERS)
portfolio_returns = compute_portfolio_returns(log_returns_df, weights)
print(portfolio_returns.head())


Date
2021-01-05    0.007759
2021-01-06   -0.024830
2021-01-07    0.023843
2021-01-08    0.006003
2021-01-11   -0.023816
Name: Portfolio, dtype: float64
